In [1]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Mar 31, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    build_feature_combos,
    detect_device,
    dropna_splits,
    load_split_bundle,
    prepare_gpu_data,
    save_colab_run,
    train_feature_sweep,
)

Mounted at /content/drive


## Load data


In [2]:
DATA_SET = 'rand_A'


df_train, df_val, df_test = load_split_bundle(CLEAN_DATA, DATA_SET)

OUTPUT_DIR = make_run_dir()

## GPU auto-detect


In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
BATCH_SIZE = CFG['BATCH']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

A100-80GB  |  VRAM: 85 GB  |  batch=65,536  |  dtype=torch.bfloat16


## Hyperparameters


In [4]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'LR={INIT_LR:.4f} (scaled from {BASE_LR} at batch {BASE_BATCH})  '
      f'BATCH={BATCH_SIZE:,}  WARMUP={WARMUP_EPOCHS} epochs')


MAX_EPOCHS=100  PATIENCE=30  LR=0.0160 (scaled from 0.001 at batch 4096)  BATCH=65,536  WARMUP=5 epochs


## Feature definitions


In [5]:
FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'iv_lag', 'd_iv_lag','vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

feature_combos = build_feature_combos(FEATURES_3, EXTRA_FEATURES, max_extra=3)

n_plus_1 = len(list(combinations(EXTRA_FEATURES, 1)))
n_plus_2 = len(list(combinations(EXTRA_FEATURES, 2)))
n_plus_3 = len(list(combinations(EXTRA_FEATURES, 3)))

print(f'Total feature combinations: {len(feature_combos)}')
print('  Base (3F):  1')
print(f'  3F + 1:     {n_plus_1}')
print(f'  3F + 2:     {n_plus_2}')
print(f'  3F + 3:     {n_plus_3}')


Total feature combinations: 130
  Base (3F):  1
  3F + 1:     9
  3F + 2:     36
  3F + 3:     84


## Drop rows with NaN and pre-allocate on GPU


In [6]:
required_cols = list(set(ALL_FEATURES + [TARGET]))
df_train, df_val, df_test, nan_stats = dropna_splits(df_train, df_val, df_test, required_cols)
print(f'Rows before: {nan_stats["before"]:,}  after: {nan_stats["after"]:,}  dropped: {nan_stats["dropped"]:,}')

gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Rows before: 3,788,641  after: 3,699,139  dropped: 89,502
Data on GPU  |  VRAM used: 0.65 GB / 85 GB  |  Free: 84.4 GB
Train: 2,589,192  Val: 739,882  Test: 370,065  Features: 12


## Analytic benchmark


In [7]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 109.5079  RMSE = 0.017202
Coefficients: a = -0.136207, b = -0.078256, c = -0.055800


## Train all feature combinations


In [8]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

print(f'\n{"=" * 70}')
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(f'{"=" * 70}\n')

all_results, elapsed_s = train_feature_sweep(
    feature_combos,
    col_idx=COL_IDX,
    train_kwargs=train_kw,
    print_every=25,
)

print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 130 MODELS
  GPU: A100-80GB  |  Batch: 65,536  |  AMP: True  |  Max epochs: 100

  [  1/130] 3F                             SSE=100.4838  Gain=+8.2%  ep=100  19.8s  elapsed=0.3min
  [ 25/130] 3F+iv_lag+rho                  SSE=62.1744  Gain=+43.2%  ep=100  12.9s  elapsed=5.4min
  [ 50/130] 3F+vix_lag+iv_lag+gamma        SSE=45.5433  Gain=+58.4%  ep=100  13.0s  elapsed=10.8min
  [ 75/130] 3F+iv_lag+d_iv_lag+vix_mom_lag SSE=46.7773  Gain=+57.3%  ep=100  12.8s  elapsed=16.2min
  [100/130] 3F+d_iv_lag+vix_mom_lag+rho    SSE=63.8884  Gain=+41.7%  ep=100  12.9s  elapsed=21.6min
  [125/130] 3F+vix_mom+theta+rho           SSE=88.2594  Gain=+19.4%  ep=100  12.9s  elapsed=26.9min
  [130/130] 3F+theta+vega+rho              SSE=92.5475  Gain=+15.5%  ep=100  13.0s  elapsed=28.0min

Done: 28.0 min for 130 models (avg 12.9s/model)


## Results summary


In [9]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,109.507927,0.000296,0.017202,0.006159,0.001515,0.002520,0.078842,None,None,None
1,3F,100.483818,0.000272,0.016478,0.007064,0.000117,0.003796,0.154751,13.8s,8.24%,None
2,3F+vix_lag,96.334023,0.000260,0.016134,0.007432,0.000313,0.004260,0.189658,12.8s,12.03%,4.13%
3,3F+iv_lag,66.274605,0.000179,0.013382,0.007095,-0.000289,0.004240,0.442512,12.7s,39.48%,31.20%
4,3F+d_iv_lag,68.196495,0.000184,0.013575,0.007006,0.000182,0.004130,0.426345,12.7s,37.72%,-2.90%
...,...,...,...,...,...,...,...,...,...,...,...
127,3F+gamma+theta+vega,91.071381,0.000246,0.015687,0.007186,-0.000425,0.004025,0.233926,12.8s,16.84%,1.89%
128,3F+gamma+theta+rho,90.119286,0.000244,0.015605,0.007287,-0.000268,0.004156,0.241935,12.9s,17.71%,1.05%
129,3F+gamma+vega+rho,105.125359,0.000284,0.016854,0.008367,0.000166,0.005235,0.115707,12.9s,4.00%,-16.65%
130,3F+theta+vega+rho,92.547523,0.000250,0.015814,0.007195,0.000247,0.004028,0.221509,12.9s,15.49%,11.96%


## Top 10 by Gain vs Hull-White


In [10]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,3F+vix_lag+iv_lag+d_iv_lag,6,40.2472,0.010429,63.25,12.9
1,3F+iv_lag+d_iv_lag+theta,6,41.6491,0.010609,61.97,12.8
2,3F+iv_lag+gamma+rho,6,42.5658,0.010725,61.13,12.8
3,3F+iv_lag+d_iv_lag+vix_mom,6,42.9891,0.010778,60.74,12.9
4,3F+iv_lag+d_iv_lag+gamma,6,43.1189,0.010794,60.62,12.9
5,3F+iv_lag+d_iv_lag+rho,6,43.5560,0.010849,60.23,13.0
6,3F+iv_lag+gamma+theta,6,44.5164,0.010968,59.35,13.0
7,3F+iv_lag+d_iv_lag+vega,6,45.1392,0.011044,58.78,12.9
8,3F+iv_lag+d_iv_lag,5,45.2150,0.011054,58.71,12.8
9,3F+vix_lag+iv_lag+gamma,6,45.5433,0.011094,58.41,13.0


## Best per group (3F, +1, +2, +3)


In [11]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f'{nf_label}: {best["combo_name"]}')
    print(f'    SSE={best["SSE"]:.4f}  RMSE={best["RMSE"]:.6f}  Gain={best["Gain_vs_HW_%"]:.2f}%\n')

3F (base): 3F
    SSE=100.4838  RMSE=0.016478  Gain=8.24%

+1 (4F): 3F+iv_lag
    SSE=66.2746  RMSE=0.013382  Gain=39.48%

+2 (5F): 3F+iv_lag+d_iv_lag
    SSE=45.2150  RMSE=0.011054  Gain=58.71%

+3 (6F): 3F+vix_lag+iv_lag+d_iv_lag
    SSE=40.2472  RMSE=0.010429  Gain=63.25%



## Summary statistics


In [12]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 27.9 min (0.46 hr)
Models trained: 130
Best overall: 3F+vix_lag+iv_lag+d_iv_lag (Gain=63.25%)


## Cleanup


In [13]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.82 GB / 85 GB
Total training time: 27.9 min for 130 models
